# Plotting PyPSA networks along the workflow

### Jupyter scripts need to be improved

In this script, the ouputs of the PyPSA networks calculated in the Data Workflow are computed.
In particular, the considered outputs are of the following scripts:
- download_osm_data
- clean_osm_data
- osm_build_network
- base_network
- add_electricity
- simplify_network
- cluster_network

In the following, the output networks described by the files are loaded and plotted by using `hvplot` package.
Such package enablKes a more interactive scouting of the files, which simplifies the learning phase and debugging process, when needed.

## Change the current directory to the main package folder

In [1]:
# change current directory to parent folder
import os
import sys
from pathlib import Path

# if not os.path.isdir("pypsa-earth"):
#     os.chdir("../..")
# sys.path.append(os.getcwd()+"/pypsa-earth/scripts")

if not Path("pypsa-earth").is_dir():
    os.chdir("../..")

sys.path.append(str(Path("pypsa-earth/scripts").resolve()))


## Specify file paths of the files related to the scripts

In [2]:
scenario_name = "KE_1"  # scenario name, default value is "" for tutorial or default configuration
                    # value shall be non null if a scenario name is specified under the "run" tag in the config file
                    # solar as non extendable carrier and non extendable in the renewables settings

# scenario_noRE = "KE_hydro_runofriver"

# check debug_base_network
# debug_base_network = (os.getcwd() + "\pypsa-earth\debug_base_network.nc")
# print(debug_base_network)

# scenario_subpath = scenario_name + "/" if scenario_name else ""
scenario_subpath = f"{scenario_name}/" if scenario_name else ""

# file paths of download_osm_data outputs
substations_ODD_path = (
    # os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "osm/raw/all_raw_substations.geojson"
    Path.cwd() /"pypsa-earth"/"resources"/ scenario_subpath /"osm"/"raw"/"all_raw_substations.geojson"
)

# print(substations_ODD_path)

lines_ODD_path = os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "osm/raw/all_raw_lines.geojson"
# file paths of clean_osm_data outputs
substations_ODC_path = (
    os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "osm/clean/all_clean_substations.geojson"
)
# print(lines_ODD_path)

lines_ODC_path = os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "osm/clean/all_clean_lines.geojson"
# file paths of osm_build_network outputs
substations_OBN_path = (
    os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "base_network/all_buses_build_network.csv"
)
lines_OBN_path = (
    os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "base_network/all_lines_build_network.csv"
)
transformer_OBN_path = (
    os.getcwd() + "/pypsa-earth/resources/" + scenario_subpath + "base_network/all_transformers_build_network.csv"
)
# base_network file path
network_path_b = os.getcwd() + "/pypsa-earth/networks/" + scenario_subpath + "base.nc"
# add_electricity file path
network_path_a = os.getcwd() + "/pypsa-earth/networks/" + scenario_subpath + "elec.nc"
# print(network_path_a)
# add_electricity for example without any custom-RE
network_path_noRE = os.getcwd() + "/pypsa-earth/networks/KE_hydro_runofriver/elec.nc"
# add_imports file path
network_path_i = os.getcwd() + "/pypsa-earth/networks/" + scenario_subpath + "elec_custom.nc"
# simplify_network file path
network_path_s = os.getcwd() + "/pypsa-earth/networks/" + scenario_subpath + "elec_s.nc"
# cluster_network file path
network_path_c = os.getcwd() + "/pypsa-earth/networks/" + scenario_subpath + "elec_s.nc"

## Load python packages

In [3]:
import logging
import os

import pypsa
import yaml
import pandas as pd
import geopandas as gpd
import geoviews as gv
import hvplot.pandas
import numpy as np
import scipy as sp
import networkx as nx
import matplotlib as plt
import holoviews as hv
# import fiona

from scipy.sparse import csgraph
from itertools import product

from shapely.geometry import Point, LineString
import shapely, shapely.prepared
from shapely.wkt import loads
import shapely
import warnings
from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) # Ignore Shapely warnings

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 70)


## Plot network of download_osm_data

download_osm_data is the script that downloads the raw OSM data
In particular, the main outputs of the script, being the substations and lines data, are loaded and then plotted.
- `df_substations_osm_raw`: is the geodataframe related to the substations
- `df_lines_osm_raw`: is the geodataframe related to the lines

The package `hvplot` enables to zoom in the image and enables to better investigate the result of the analysis.
Therefore, it has been used as detailed below.

In [4]:
# load substation geodataframe
# print(substations_ODD_path)

# with fiona.open(substations_ODD_path) as f:
#     print(f.schema)  # zeigt dir den Feldtyp, der Probleme macht, wird aber eh geskippt

df_substations_osm_raw = gpd.read_file(substations_ODD_path)
# load lines geodataframe
df_lines_osm_raw = gpd.read_file(lines_ODD_path)

print(df_substations_osm_raw.geom_type.value_counts())
df_substations_osm_raw_points = df_substations_osm_raw[df_substations_osm_raw.geom_type == "Point"] # error that mixed geadatatyes.
df_substations_osm_raw_polygons = df_substations_osm_raw[df_substations_osm_raw.geom_type == "Polygon"] # not necessary

# hvplot
hhosmraw = df_substations_osm_raw_points.hvplot(
    geo=True,
    size=10,  # buses["tag_area"]**(0.5)/10,
    frame_height=750,
    frame_width=750,
    alpha=0.4,
    tiles="CartoLight",
    color="orange",
) * df_lines_osm_raw.hvplot(geo=True, alpha=0.4,).opts(
    active_tools=["pan", "wheel_zoom"]
)
display (hhosmraw)     # show plot in the notebook
hv.save(hhosmraw, "documentation/osm_raw_network.html"); # save plot in the path here

c:\Users\Marie\anaconda3\envs\pypsa-earth\Lib\site-packages\pyogrio\__init__.py:7: DeprecationWarning: The 'shapely.geos' module is deprecated, and will be removed in a future version. All attributes of 'shapely.geos' are available directly from the top-level 'shapely' namespace (since shapely 2.0.0).
  import shapely.geos  # noqa: F401
Skipping field refs: unsupported OGR type: 13
Skipping field refs: unsupported OGR type: 13


Polygon    238
Point       20
Name: count, dtype: int64


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]

## Plot network of clean_osm_data

clean_osm_data is the script that loads the raw OSM data and performs simple cleaning, such as renaming columns to adhere with PyPSA format, cleaning Nones/NaNs, etc.
In particular, the main outputs of the script, being the substations and lines data, are loaded and then plotted.
- `df_substations_osm_clean`: is the geodataframe related to the substations
- `df_lines_osm_clean`: is the geodataframe related to the lines

The package `hvplot` enables to zoom in the image and enables to better investigate the result of the analysis.
Therefore, it has been used as detailed below.

In [5]:
# load substation geodataframe
df_substations_osm_clean = gpd.read_file(substations_ODC_path)
# load lines geodataframe
df_lines_osm_clean = gpd.read_file(lines_ODC_path)

print(df_substations_osm_clean.geom_type.value_counts())

# hvplot
hhosm = df_substations_osm_clean.hvplot(
    geo=True,
    size=10,  # buses["tag_area"]**(0.5)/10,
    frame_height=750,
    frame_width=750,
    alpha=0.4,
    tiles="CartoLight",
    color="orange",
    # hover_cols=['bus_id'],
) * df_lines_osm_clean.hvplot(
    geo=True,
    alpha=0.4,
    # hover_cols=['line_id'],
).opts(
    active_tools=["pan", "wheel_zoom"]
)
display (hhosmraw)     # show plot in the notebook
hv.save(hhosm, "documentation/osm_clean_network.html"); # save plot in the path here

Point    470
Name: count, dtype: int64


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]

## Plot network of osm_build_network

osm_build_network is the script that loads the cleaned OSM data and performs some refinings to prepare the input data that are loaded by the build_base_network
In particular, the main outputs of the script, being the substations and lines data, are loaded and then plotted.
- `df_substations_osm_build`: is the geodataframe related to the substations
- `df_lines_osm_build`: is the geodataframe related to the lines

The package `hvplot` enables to zoom in the image and enables to better investigate the result of the analysis.
Therefore, it has been used as detailed below.

In [6]:
# load substations
substations_OBN = gpd.read_file(
    substations_OBN_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO"
)
substations_OBN["geometry"] = gpd.points_from_xy(
    pd.to_numeric(substations_OBN.lon, downcast="float"),
    pd.to_numeric(substations_OBN.lat, downcast="float"),
)
substations_OBN = substations_OBN.set_crs(epsg=4326, inplace=True)
# load lines
lines_OBN = gpd.read_file(
    lines_OBN_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO"
).set_crs(epsg=4326, inplace=True)

# # print(lines_OBN)
# filtered_df = lines_OBN[lines_OBN["dc"] == "True"]
# print(filtered_df)

# print(substations_OBN.iloc[85]) # substation close to the border to Ethiopia

# hvplot
hho = substations_OBN.hvplot(
    geo=True,
    size=10,  # buses["tag_area"]**(0.5)/10,
    frame_height=700,
    frame_width=700,
    alpha=0.4,
    tiles="CartoLight",
    color="orange",
    # hover_cols=['bus_id']
) * lines_OBN.hvplot(
    geo=True,
    alpha=0.4,
    # hover_cols='line_id'
).opts(
    active_tools=["pan", "wheel_zoom"]
)
display(hho)
hv.save(hho, "documentation/osm_build_network.html");



:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]

## Plot network of base_network

`base_network` is the script that loads the cleaned OSM data, thus the map shown above, and prepare a preliminary PyPSA model that is later used by all scripts downstream the workflow.
In particular, in order to compare the network structure, the substations and lines datasets are loaded
- `n_b` is the main PyPSA network model related to `base_network`
- `n_b.buses` is the geodataframe related to the substations
- `n_b.lines` is the geodataframe related to the lines

The package `hvplot` is used to show the model.

In [7]:
# base_network PyPSA model
n_b = pypsa.Network(network_path_b)

# lines dataframe
lines_b = n_b.lines
lines_b["geometry"] = lines_b["geometry"].apply(loads)
lines_b = gpd.GeoDataFrame(lines_b, crs="epsg:4326")

# buses dataframe
buses_b = n_b.buses
buses_b["geometry"] = gpd.points_from_xy(buses_b.lon, buses_b.lat)
buses_b = gpd.GeoDataFrame(buses_b, crs="epsg:4326")

# transformer dataframe
transformer_b = gpd.read_file(
    transformer_OBN_path, GEOM_POSSIBLE_NAMES="geometry", KEEP_GEOM_COLUMNS="NO"
)

# hvplot
hhb = (
    buses_b.hvplot(
        geo=True,
        size=10,  # buses["tag_area"]**(0.5)/10,
        frame_height=750,
        frame_width=750,
        alpha=0.4,
        tiles="CartoLight",
        color="orange",
    )
    * lines_b.hvplot(geo=True, alpha=0.4).opts(active_tools=["pan", "wheel_zoom"])
    * transformer_b.hvplot(geo=True, color="red", alpha=0.4).opts(
        active_tools=["pan", "wheel_zoom"]
    )
)
display (hhb)
hv.save(hhb, "documentation/base_network.html");

# what happened to the line connection to Ethiopia?? - it is in the input so it has to be some configuration.
# why is that line gone????? - hvdc_as_line was set to false - then I can's set it manually and hvdc line is modelled as link. that is why it disappeared in my plot
# import still impossible. 
# has to be in the osm data set...

Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'transf_15_0', 'transf_15_1', 'transf_16_0', 'transf_17_0',
       'transf_18_0', 'transf_22_0', 'transf_22_1', 'transf_24_0',
       'transf_26_0', 'transf_27_0', 'transf_32_0', 'transf_34_0',
       'transf_69_0'],
      dtype='object', name='name')
Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'transf_15_0', 'transf_15_1', 'transf_16_0', 'transf_17_0',
       'transf_18_0', 'transf_22_0', 'transf_22_1', 'transf_24_0',
       'transf_26_0', 'transf_27_0', 'transf_32_0', 'transf_34_0',
       'transf_69_0'],
      dtype='object', name='name')
Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'tra

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]
   .Path.II  :Path   [Longitude,Latitude]

In [8]:
# print(n_b.links)
print(n_b.transformers)
print(n_b.buses.v_nom.unique())


             Unnamed: 0 bus0 bus1  voltage_bus0  voltage_bus1 country  \
Transformer                                                             
transf_0_0           92    0    1        132000        220000      KE   
transf_2_0           93    3    4        132000        220000      KE   
transf_4_0           94    6    7        132000        220000      KE   
transf_5_0           95    8    9         66000        220000      KE   
transf_6_0           96   10   11        132000        220000      KE   
transf_7_0           97   12   13         33000        400000      KE   
transf_8_0           98   14   15        220000        400000      KE   
transf_10_0          99   17   18         33000        132000      KE   
transf_10_1         100   18   19        132000        220000      KE   
transf_12_0         101   21   22         33000        132000      KE   
transf_15_0         102   25   26         33000        132000      KE   
transf_15_1         103   26   27        132000    

In [9]:
warned = [
 "transf_0_0","transf_2_0","transf_4_0","transf_5_0","transf_6_0","transf_7_0",
 "transf_8_0","transf_10_0","transf_10_1","transf_12_0","transf_15_0","transf_15_1",
 "transf_16_0","transf_17_0","transf_18_0","transf_22_0","transf_22_1","transf_24_0",
 "transf_26_0","transf_27_0","transf_32_0","transf_34_0","transf_69_0"
]
set(warned) # - set(n_b.transformers.index)  # if transformers remain here, they were dropped

n_b.transformers.index


Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'transf_15_0', 'transf_15_1', 'transf_16_0', 'transf_17_0',
       'transf_18_0', 'transf_22_0', 'transf_22_1', 'transf_24_0',
       'transf_26_0', 'transf_27_0', 'transf_32_0', 'transf_34_0',
       'transf_69_0'],
      dtype='object', name='Transformer')

## Plot network of add_electricity

`add_electricity` adds generators and load to the `base_network` model, as such no differences are expected with respect to the output model of `base_network`

Similarly to the previous cases:
- `n_a` is the PyPSA network model
- `n_a.lines` is the lines dataframe
- `n_a.buses` is the buses dataframe

In [10]:
bad = (~n_b.transformers.bus0.isin(n_b.buses.index)) | (~n_b.transformers.bus1.isin(n_b.buses.index))
print("Transformers referencing non-existent buses *now*:", int(bad.sum()))
print("Any NaN bus references in transformers:", n_b.transformers[['bus0','bus1']].isna().any().any())


Transformers referencing non-existent buses *now*: 0
Any NaN bus references in transformers: False


In [11]:
# elec.nc is the network at 2030 including generation, demand,... with build renewable profiles,...
n_a = pypsa.Network(network_path_a)

# lines dataframe
lines_a = n_a.lines
lines_a["geometry"] = lines_a["geometry"].apply(loads)
lines_a = gpd.GeoDataFrame(lines_a, crs="epsg:4326")
lines_a["line_id"] = pd.Series(lines_a.index.map(str), index=lines_a.index)

# buses dataframe
buses_a = n_a.buses
buses_a["geometry"] = gpd.points_from_xy(buses_a.lon, buses_a.lat)
buses_a = gpd.GeoDataFrame(buses_a, crs="epsg:4326")

links_a = n_a.links
links_a["geometry"] = links_a["geometry"].apply(loads)
links_a = gpd.GeoDataFrame(links_a, crs="epsg:4326")
links_a["link_id"] = pd.Series(links_a.index.map(str), index=links_a.index)

# print(n_a.generators.carrier.value_counts())
# print(n_a.generators.loc[:, ["bus", "carrier", "p_nom"]])

# hvplot
hha = (
    buses_a.hvplot(
        geo=True,
        size=10,  # buses["tag_area"]**(0.5)/10,
        frame_height=750,
        frame_width=750,
        alpha=0.4,
        tiles="CartoLight",
        color="orange",
    )
    * lines_a.hvplot(geo=True, alpha=0.4, hover_cols="line_id")
    * links_a.hvplot(geo=True, alpha=0.4, color="green", hover_cols="link_id"
    # * lines_a.loc[["8"]].hvplot(
    #     geo=True,
    #     alpha=0.9,
    #     color="red",
    #     line_width=2,
    #     hover_cols="line_id"
    ).opts(active_tools=["pan", "wheel_zoom"])
    # * buses_a.loc[["46"]].hvplot(
    #     geo=True,
    #     alpha=0.9,
    #     color="red",
    #     size=10
    # )           
    * buses_a.loc[["85"]].hvplot(
        geo=True,
        alpha=0.9,
        color="green",
        size=5
    )
)
display (hha)
hv.save(hha, "documentation/add_electricity.html")

Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'transf_15_0', 'transf_15_1', 'transf_16_0', 'transf_17_0',
       'transf_18_0', 'transf_22_0', 'transf_22_1', 'transf_24_0',
       'transf_26_0', 'transf_27_0', 'transf_32_0', 'transf_34_0',
       'transf_69_0'],
      dtype='object', name='name')
Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'transf_15_0', 'transf_15_1', 'transf_16_0', 'transf_17_0',
       'transf_18_0', 'transf_22_0', 'transf_22_1', 'transf_24_0',
       'transf_26_0', 'transf_27_0', 'transf_32_0', 'transf_34_0',
       'transf_69_0'],
      dtype='object', name='name')
Index(['transf_0_0', 'transf_2_0', 'transf_4_0', 'transf_5_0', 'transf_6_0',
       'transf_7_0', 'transf_8_0', 'transf_10_0', 'transf_10_1', 'transf_12_0',
       'tra

:Overlay
   .WMTS.I    :WMTS   [Longitude,Latitude]
   .Points.I  :Points   [Longitude,Latitude]
   .Path.I    :Path   [Longitude,Latitude]   (line_id)
   .Path.II   :Path   [Longitude,Latitude]   (link_id)
   .Points.II :Points   [Longitude,Latitude]

In [12]:
# wo ist meine Wanjii power station? in powerplants.csv ist sie drinnen.

# print(n_a.buses)
# print(n_a.generators)
print(n_a.generators[(n_a.generators["carrier"] == "geothermal") & (n_a.generators["p_nom_min"] != 0)])
# print(n_a.generators[(n_a.generators["carrier"] == "solar")])
# print(n_a.generators[n_a.generators["bus"]=="84"])
# print(n_noRE.generators[n_noRE.generators["bus"]=="58"])
# border_bus= n_a.buses[n_a.buses["carrier"] == "DC"]
# print(border_bus)
# print(n_a.carriers)
# print(n_noRE.carriers)

print(n_a.generators.groupby("carrier").p_nom.sum())
# print(n_noRE.generators.groupby("carrier").p_nom.sum())

# hydro_test
# n_a.add



                                            carrier  bus  p_nom_min  p_nom  \
Generator                                                                    
CAkiira Geothermal                       geothermal   10       70.0   70.0   
CBaringo GDC Project Korosi              geothermal  103      100.0  100.0   
CBaringo GDC Project Paka                geothermal  103      100.0  100.0   
CBaringo GDC Project Silali              geothermal  103      100.0  100.0   
CBVC geothermal (Frontier)               geothermal    5      140.0  140.0   
CEburru Wellhead                         geothermal   61        2.5    2.5   
CMenengai geothermal plant (Globeleq)    geothermal   98       35.0   35.0   
CMenengai geothermal plant (Sosian)      geothermal   98       35.0   35.0   
CMenengai geothermal plant (OrPower 22)  geothermal   98       35.0   35.0   
COlkaria 1                               geothermal   90       45.0   45.0   
COlkaria 1 units 4 & 5                   geothermal   10      14

In [13]:
# # elec.nc is the network at 2030 including generation, demand,... with build renewable profiles,...
# # hier

# n_i = pypsa.Network(network_path_i)

# # lines dataframe
# lines_i = n_i.lines
# lines_i["geometry"] = lines_i["geometry"].apply(loads)
# lines_i = gpd.GeoDataFrame(lines_i, crs="epsg:4326")
# lines_i["line_id"] = pd.Series(lines_i.index.map(str), index=lines_a.index)

# # links dataframe
# # links_i = n_i.links
# # links_i["geometry"] = links_i["geometry"].apply(loads)
# # links_i = gpd.GeoDataFrame(links_i, crs="epsg:4326")
# # links_i["link_id"] = pd.Series(links_i.index.map(str), index=links_i.index)

# # n_i.buses.loc["import_from_ethiopia", "under_construction"] = False
# n_i.buses.loc["import_from_ethiopia", "lon"] = float(38.0)
# n_i.buses.loc["import_from_ethiopia", "lat"] = float(4.0)

# # buses dataframe
# buses_i = n_i.buses
# buses_i["geometry"] = gpd.points_from_xy(buses_i.lon, buses_i.lat)
# buses_i = gpd.GeoDataFrame(buses_i, crs="epsg:4326")



# # print(n_i.buses.loc[["import_from_ethiopia"]])
# # print(n_i.buses)

# # print(n_i.generators.carrier.value_counts())
# # print(n_i.generators.loc[:, ["bus", "carrier", "p_nom"]])

# # hvplot
# hhi = (
#     buses_i.hvplot(
#         geo=True,
#         size=10,  # buses["tag_area"]**(0.5)/10,
#         frame_height=750,
#         frame_width=750,
#         alpha=0.4,
#         tiles="CartoLight",
#         color="orange",
#     )
#     * lines_i.hvplot(geo=True, alpha=0.4, hover_cols="line_id")
#     # * links_i.hvplot(geo=True, alpha=0.4, color="blue", hover_cols="link_id")
#     # * lines_a.loc[["8"]].hvplot(
#     #     geo=True,
#     #     alpha=0.9,
#     #     color="red",
#     #     line_width=2,
#     #     hover_cols="line_id"
#     # ).opts(active_tools=["pan", "wheel_zoom"]
#     #      )
# )
# display (hhi)
# hv.save(hhi, "documentation/add_imports.html");

In [14]:
# print(n_i.links)

# print(n_i.buses.loc["import_from_ethiopia"])

# # change information of one of the buses:
# n_i.buses.loc["import_from_ethiopia", "under_construction"] = False
# n_i.buses.loc["import_from_ethiopia", "lon"] = float(38.0)
# n_i.buses.loc["import_from_ethiopia", "lat"] = float(4.5)

# print(n_i.buses.loc["import_from_ethiopia"])



## Plot network of simplify_network

`simplify_network` modifies the structure of the network obtained from `add_electricity` to remove dead-hends and group disconnected nodes that are not connected with the main network.
The differences between the networks by `simplify_network` and `add_electricity` can be easily scouted by comparing the hvplots of the previous cell and the following.

Similarly to the previous cases:
- `n_s` is the PyPSA network model
- `n_s.lines` is the lines dataframe
- `n_s.buses` is the buses dataframe

In [15]:
n_s = pypsa.Network(network_path_s)

# buses dataframe
buses_s = n_s.buses
buses_s["geometry"] = gpd.points_from_xy(buses_s.x, buses_s.y)
buses_s = gpd.GeoDataFrame(buses_s, crs="epsg:4326")

# lines dataframe
lines_s = n_s.lines
lines_s["geometry"] = lines_s["geometry"] = lines_s.apply(
    lambda x: LineString(
        [buses_s.loc[x["bus0"], "geometry"], buses_s.loc[x["bus1"], "geometry"]]
    ),
    axis=1,
)
lines_s = gpd.GeoDataFrame(lines_s, crs="epsg:4326")
lines_s["line_id"] = pd.Series(lines_s.index.map(str), index=lines_s.index)

# hvplot
hhs = (
    buses_s.hvplot(
        geo=True,
        size=10,  # buses["tag_area"]**(0.5)/10,
        frame_height=750,
        frame_width=750,
        alpha=0.4,
        tiles="CartoLight",
        color="orange",
    )
    * lines_s.hvplot(geo=True, alpha=0.4, hover_cols="line_id").opts(active_tools=["pan", "wheel_zoom"])
    * buses_s.loc[["45"]].hvplot(
        geo=True,
        alpha=0.9,
        color="green",
        size=5
    )
)

display(hhs)
hv.save(hhs, "documentation/simplified_cluster.html")

# Lake Turkana wind turbine are cut off of the grid (dead end?), LTWT is around 17% of the countries installed capacity. Can't be cutoff.

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Marie\\Semesterthesis\\PyPSA-Earth-Project\\pypsa-earth\\networks\\KE_1\\elec_s.nc'

In [ ]:
ids_220 = lines_a[lines_a.v_nom == 220].index
lines_a.loc[ids_220, "s_nom"]
ids_220_in_s = ids_220.intersection(lines_s.index)
check_df = pd.DataFrame({"simplified": lines_s.loc[ids_220_in_s, "s_nom"], "add": lines_a.loc[ids_220_in_s, "s_nom"]})
max_delta = (check_df["simplified"] - check_df["add"]).abs().max()
max_delta # highest discrepancy for the simplified network and a more detailed one (elec.nc) - e.g 400kV line from Lake turkana is being removed, here they only compare lines that are in both networks but the capacities might change due to the removel of the 400 kV line

print(n_s.generators[(n_s.generators["carrier"] == "solar")])
print(n_s.generators.groupby("carrier").p_nom.sum())


          carrier bus  p_nom_min  p_nom  p_nom_extendable  efficiency  \
Generator                                                               
1 solar     solar   1        0.0    0.0              True         1.0   
11 solar    solar  11        0.0    0.0              True         1.0   
15 solar    solar  15        0.0    0.0              True         1.0   
16 solar    solar  16        0.0    0.0              True         1.0   
2 solar     solar   2        0.0    0.0              True         1.0   
20 solar    solar  20        0.0    0.0              True         1.0   
24 solar    solar  24        0.0    0.0              True         1.0   
27 solar    solar  27        0.0    0.0              True         1.0   
34 solar    solar  34        0.0    0.0              True         1.0   
36 solar    solar  36        0.0    0.0              True         1.0   
39 solar    solar  39        0.0    0.0              True         1.0   
4 solar     solar   4        0.0    0.0            

In [ ]:
# check for the bad line.

check_df["delta"] = (check_df["simplified"] - check_df["add"]).abs()
check_df.sort_values("delta", ascending=False).head(5)

worst_line = check_df.sort_values("delta", ascending=False).index[0]
# print(lines_a.loc[worst_line])  # augmented data
# print(lines_s.loc[worst_line])  # simplified data

check_df.loc[check_df["delta"] == max_delta]

# print(lines_s.loc[["8"]])




,simplified,add,delta
Line,,,
1,1474.668058,1474.668058,0.0
3,2949.336115,2949.336115,0.0
4,2949.336115,2949.336115,0.0
5,2949.336115,2949.336115,0.0
8,1474.668058,1474.668058,0.0
16,1474.668058,1474.668058,0.0
19,1474.668058,1474.668058,0.0
23,1474.668058,1474.668058,0.0
35,1474.668058,1474.668058,0.0


## Plot network of cluster_network

`cluster_network` modifies the structure of the network obtained from `simplify_network` to aggregate close nodes and reduce the size of the problem as specified by the configuration file ``config.yaml``.
The differences between the networks by `cluster_network` and `simplify_network` can be easily scouted by comparing the hvplots of the previous cell and the following.

Similarly to the previous cases:
- `n_c` is the PyPSA network model
- `n_c.lines` is the lines dataframe
- `n_c.buses` is the buses dataframe

In [ ]:
n_c = pypsa.Network(network_path_c)

# buses dataframe
buses_c = n_c.buses
buses_c["geometry"] = gpd.points_from_xy(buses_c.x, buses_c.y)
buses_c = gpd.GeoDataFrame(buses_c, crs="epsg:4326")

# lines dataframe
lines_c = n_c.lines
lines_c["geometry"] = lines_c.apply(
    lambda x: LineString(
        [buses_c.loc[x["bus0"], "geometry"], buses_c.loc[x["bus1"], "geometry"]]
    ),
    axis=1,
)
lines_c = gpd.GeoDataFrame(lines_c, crs="epsg:4326")

# hvplot
hhc = buses_c.hvplot(
    geo=True,
    size=10,  # buses["tag_area"]**(0.5)/10,
    frame_height=750,
    frame_width=750,
    alpha=0.4,
    tiles="CartoLight",
    color="orange",
) * lines_c.hvplot(geo=True, alpha=0.4).opts(active_tools=["pan", "wheel_zoom"])
display(hhc)
hv.save(hhc, "documentation/network_cluster.html")


INFO:pypsa.io:Imported network elec_s.nc has buses, carriers, generators, lines, loads


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]
   .Path.I   :Path   [Longitude,Latitude]